# Data analysis

A walk through the raw data before any modelling: volumes, the event funnel,
platform structure, sparsity and the long tail. It only reads `data/`, so it
reproduces from the mounted CSVs. Run top to bottom.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 60)

# data/ lives at the repo root; work whether the kernel was started there or here
root = Path.cwd()
if not (root / "data").exists():
    root = root.parent
DATA = root / "data"

users = pd.read_csv(DATA / "users.csv")
apps = pd.read_csv(DATA / "apps.csv")
inter = pd.read_csv(DATA / "interactions.csv")
inter["event_time"] = pd.to_datetime(inter["event_time"], errors="coerce", format="mixed")

## Volumes

In [ ]:
n_users = users["user_id"].nunique()
n_apps = apps["app_id"].nunique()
n_u = inter["user_id"].nunique()
n_a = inter["app_id"].nunique()
n_pairs = inter.groupby(["user_id", "app_id"]).ngroups

print("interactions rows    :", len(inter))
print("distinct users in log:", n_u, "(users.csv has", n_users, ")")
print("distinct apps in log :", n_a, "(apps.csv has", n_apps, ")")
print("distinct (user,app)  :", n_pairs)

The log references more users and apps than exist in their own tables - 244
extra users, 185 extra apps. Those are the cold-start orphans flagged in the
data-quality report.

## Event funnel

In [ ]:
order = ["af_view", "af_click", "af_install", "af_open",
         "af_complete_registration", "af_level_achieved", "af_purchase"]
tot = len(inter)
mix = inter["event_name"].value_counts().reindex(order).fillna(0).astype(int)
mix_pct = (mix / tot * 100).round(1)

print(pd.DataFrame({"count": mix, "share_%": mix_pct}).to_string())
print()
print("view + click share:", round((mix["af_view"] + mix["af_click"]) / tot * 100, 1), "%")
print("install-or-beyond :", round((tot - mix["af_view"] - mix["af_click"]) / tot * 100, 1), "%")

The log is top-heavy: 64.1% of events are `af_view` + `af_click`. That matters
for evaluation, because a popularity list scores cheap hits on exactly those.

## Platform

In [ ]:
print("users by platform:\n", users["platform"].value_counts().to_string())
print("\napps by platform:\n", apps["platform"].value_counts().to_string())

m = (inter[["user_id", "app_id"]]
     .merge(users[["user_id", "platform"]].rename(columns={"platform": "uplat"}), on="user_id")
     .merge(apps[["app_id", "platform"]].rename(columns={"platform": "aplat"}), on="app_id"))
cross = (m["uplat"] != m["aplat"]).sum()
print(f"\ncross-platform rows: {cross:,} ({cross / len(inter) * 100:.1f}%)")

## Sparsity

In [ ]:
pairs = inter.groupby(["user_id", "app_id"]).ngroups
possible = n_u * n_a
print(f"observed pairs : {pairs:,}")
print(f"possible pairs : {possible:,}")
print(f"density        : {pairs / possible * 100:.2f}%")

## Long tail

In [ ]:
app_users = inter.groupby("app_id")["user_id"].nunique()
print("apps touched by exactly 1 user :", (app_users == 1).sum())
print("apps touched by <= 5 users     :", (app_users <= 5).sum())
print("median distinct users per app  :", int(app_users.median()))
print("max distinct users per app     :", int(app_users.max()))

## User activity

In [ ]:
n_inter = inter.groupby("user_id").size()
n_apps_per_user = inter.groupby("user_id")["app_id"].nunique()
print("interactions per user  : median", int(n_inter.median()),
      " mean", round(n_inter.mean(), 1), " max", int(n_inter.max()))
print("distinct apps per user : median", int(n_apps_per_user.median()),
      " mean", round(n_apps_per_user.mean(), 1), " max", int(n_apps_per_user.max()))
print("users with <5 events   :", int((n_inter < 5).sum()))

## Time window and catalog dimensions

In [ ]:
print("window:", inter["event_time"].min().date(), "->", inter["event_time"].max().date(),
      f"({(inter['event_time'].max() - inter['event_time'].min()).days} days)")

print("\napp status:\n", apps["status"].value_counts().to_string())
print("\napp category (top 10):\n", apps["category"].value_counts().head(10).to_string())
print("\nage bucket:\n", users["age_bucket"].value_counts().to_string())
print("\ncountry (top 10):\n", users["country_code"].value_counts().head(10).to_string())
print("\nnull country_code:", int(users["country_code"].isna().sum()),
      "| null duration_seconds:", int(inter["duration_seconds"].isna().sum()))

## Takeaway

Small, sparse, tabular, long-tail-heavy implicit feedback, gated by platform,
with a strong prior that same-platform and already-liked category dominate.
That shape points at a feature-based GBDT rather than a latent model, and the
245 single-touch apps put a hard ceiling on any retrieval metric. The full
write-up is in `ANALYSIS.md`.